# Prediksi Dropout Mahasiswa — Notebook Penelitian

**Arsitektur Model**: Stacking Ensemble (XGBoost + LightGBM + CatBoost + Logistic Regression) + SMOTE-ENN + Optimasi Threshold + SHAP Explainability  
**Dataset**: Higher Education Student Performance & Dropout Dataset (Klasifikasi Biner: Dropout=1 vs Graduate=0)


In [ ]:
# Tahap 0: Konfigurasi Lingkungan & Import Library
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
import shap
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef,
    precision_score, recall_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print("✅ Lingkungan eksperimen berhasil dikonfigurasi.")

## Tahap 1: Pengambilan Data, Penyaringan & Enkoding Target
- Dataset Awal: `dataset.csv` (4.424 sampel, 37 atribut)
- Penyaringan: Menghapus status `Enrolled` (status sementara) -> Klasifikasi Biner (Dropout=1 vs Graduate=0)

In [ ]:
# Menentukan jalur dataset relatif terhadap direktori notebook
DATA_PATH = os.path.join("..", "data", "raw", "dataset.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("data", "raw", "dataset.csv")

df_raw = pd.read_csv(DATA_PATH)
print(f"Ukuran dataset mentah: {df_raw.shape}")

# Penyaringan biner: Hanya mempertahankan Graduate dan Dropout
df_binary = df_raw[df_raw["Target"] != "Enrolled"].copy()
print(f"Ukuran dataset biner setelah disaring: {df_binary.shape}")

# Enkoding Target: Dropout -> 1, Graduate -> 0
df_binary["Target"] = df_binary["Target"].map({"Dropout": 1, "Graduate": 0})

X = df_binary.drop(columns=["Target"])
y = df_binary["Target"]

print("\nDistribusi Kelas Target:")
print(f"  Graduate (0): {(y == 0).sum()} ({y.value_counts(normalize=True)[0]*100:.1f}%)")
print(f"  Dropout  (1): {(y == 1).sum()} ({y.value_counts(normalize=True)[1]*100:.1f}%)")

## Tahap 2: Pembagian Data Stratified & Pembobotan Skala Fitur
- Rasio Pembagian: 80% Data Latih (2.904 sampel), 20% Data Uji (726 sampel)
- Skalasi: `StandardScaler` dilatih strictly pada data latih untuk mehindari kebocoran data (*data leakage*)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(f"Jumlah data latih: {X_train_sc.shape[0]} sampel")
print(f"Jumlah data uji:   {X_test_sc.shape[0]} sampel")

## Tahap 3: Penyeimbangan Kelas via SMOTE-ENN
- Over-sampling: `SMOTE` (k_neighbors=5, target_ratio=1.0)
- Under-sampling: `EditedNearestNeighbours` (n_neighbors=3) untuk pembersihan noise

In [ ]:
# Tahap 3: Penyeimbangan Kelas via SMOTE-ENN
# Menggunakan parameter dari config.py: sampling_strategy=0.9, kind_sel="mode"
smote = SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)
enn = EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")

X_smote, y_smote = smote.fit_resample(X_train_sc, y_train)
X_res, y_res = enn.fit_resample(X_smote, y_smote)

# Mengonversi kembali ke DataFrame untuk konsistensi kolom
X_res = pd.DataFrame(X_res, columns=X_train_sc.columns)
y_res = pd.Series(y_res, name=y_train.name if y_train.name else "Target")

print("Sebelum SMOTE-ENN:")
print(f"  Graduate (0): {(y_train == 0).sum()}")
print(f"  Dropout  (1): {(y_train == 1).sum()}")

n_res_grad = (y_res == 0).sum()
n_res_drop = (y_res == 1).sum()
rasio = n_res_drop / n_res_grad

print("\nSetelah SMOTE-ENN:")
print(f"  Graduate (0): {n_res_grad}")
print(f"  Dropout  (1): {n_res_drop}")
print(f"  Rasio        : {rasio:.3f} ({rasio*100:.1f}%)")
print(f"  Ukuran Data Latih Setelah Resampling: {X_res.shape}")

## Tahap 4: Optimasi Hyperparameter Optuna
Pencarian hyperparameter optimal bebas *leakage* menggunakan 5-Fold Stratified CV yang dibungkus dalam `ImbPipeline` (skalasi dan resampling dilakukan di dalam setiap fold).

In [ ]:
# Tahap 4: Optimasi Hyperparameter Optuna (Bebas Leakage)
# SMOTE-ENN dan StandardScaler diletakkan di dalam ImbPipeline agar fit strictly pada lipatan latih CV.
# Parameter SMOTE-ENN di CV diselaraskan: sampling_strategy=0.9, kind_sel="mode".

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'gamma': trial.suggest_float('gamma', 0.0, 10.0),
        'random_state': SEED,
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    pipe = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)),
        ('enn', EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")),
        ('clf', XGBClassifier(**params))
    ])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    return scores.mean()

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'random_state': SEED,
        'verbose': -1
    }
    pipe = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)),
        ('enn', EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")),
        ('clf', LGBMClassifier(**params))
    ])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    return scores.mean()

def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 700),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_seed': SEED,
        'verbose': False
    }
    pipe = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)),
        ('enn', EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")),
        ('clf', CatBoostClassifier(**params))
    ])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    return scores.mean()

print("Menjalankan optimasi hyperparameter Optuna untuk XGBoost, LightGBM, dan CatBoost...")
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=30, timeout=120)
best_params_xgb = study_xgb.best_params
best_params_xgb.update({'random_state': SEED, 'use_label_encoder': False, 'eval_metric': 'logloss'})

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(objective_lgbm, n_trials=30, timeout=120)
best_params_lgb = study_lgb.best_params
best_params_lgb.update({'random_state': SEED, 'verbose': -1})

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cat.optimize(objective_catboost, n_trials=20, timeout=120)
best_params_cat = study_cat.best_params
best_params_cat.update({'random_seed': SEED, 'verbose': False})

print(f"✅ Optimasi Optuna selesai.")
print(f"  Rata-rata F1 CV Terbaik (XGBoost):  {study_xgb.best_value:.4f}")
print(f"  Rata-rata F1 CV Terbaik (LightGBM): {study_lgb.best_value:.4f}")
print(f"  Rata-rata F1 CV Terbaik (CatBoost): {study_cat.best_value:.4f}")

## Tahap 5: Pelatihan Model Proposed (Stacking Ensemble)
Level 0 Base Learners: XGBoost + LightGBM + CatBoost + Logistic Regression  
Level 1 Meta Learner: Logistic Regression (dengan class_weight='balanced')  
Prediksi Out-Of-Fold (OOF) 5-Fold digunakan untuk melatih meta-learner secara adil tanpa kebocoran data.

In [ ]:
# Tahap 5: Pelatihan Model Proposed (Stacking Ensemble)
# Stacking Ensemble (LGBM + CatBoost + Logistic Regression)
# Memakai apply_resampling=True dan fit strictly pada X_train_sc (leakage-free OOF)
# XGBoost dinonaktifkan (use_xgb=False) untuk parsimoni dan penyederhanaan model.

class StackingEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, xgb_params=None, lgbm_params=None, catboost_params=None,
                 seed=SEED, apply_resampling=False, use_xgb=True, use_lr=True):
        self.xgb_params = xgb_params or {}
        self.lgbm_params = lgbm_params or {}
        self.catboost_params = catboost_params or {}
        self.seed = seed
        self.apply_resampling = apply_resampling
        self.use_xgb = use_xgb
        self.use_lr = use_lr

    def fit(self, X, y):
        X_arr = np.array(X)
        y_arr = np.array(y)
        
        self.classes_ = np.unique(y_arr)
        self.n_features_in_ = X_arr.shape[1]
        
        # Inisialisasi model
        self.xgb_ = XGBClassifier(**self.xgb_params) if self.use_xgb else None
        self.lgb_ = LGBMClassifier(**self.lgbm_params)
        self.cat_ = CatBoostClassifier(**self.catboost_params)
        self.lr_ = LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=self.seed
        ) if self.use_lr else None
        
        self.meta_learner_ = LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=self.seed
        )
        
        self.base_models = {
            'lgb': self.lgb_,
            'cat': self.cat_
        }
        if self.use_xgb:
            self.base_models['xgb'] = self.xgb_
        if self.use_lr:
            self.base_models['lr'] = self.lr_
        
        n_base = len(self.base_models)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.seed)
        oof_preds = np.zeros((X_arr.shape[0], n_base))
        
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_arr, y_arr)):
            X_tr, X_val = X_arr[train_idx], X_arr[val_idx]
            y_tr = y_arr[train_idx]
            
            # Terapkan SMOTE-ENN secara terisolasi di dalam lipatan (leakage-free)
            if self.apply_resampling:
                smote = SMOTE(k_neighbors=5, random_state=self.seed, sampling_strategy=0.9)
                enn = EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")
                X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
                X_tr, y_tr = enn.fit_resample(X_tr_res, y_tr_res)
            
            f_lgb = LGBMClassifier(**self.lgbm_params)
            f_cat = CatBoostClassifier(**self.catboost_params)
            
            f_lgb.fit(X_tr, y_tr)
            f_cat.fit(X_tr, y_tr, verbose=False)
            
            col_idx = 0
            if self.use_xgb:
                f_xgb = XGBClassifier(**self.xgb_params)
                f_xgb.fit(X_tr, y_tr, verbose=False)
                oof_preds[val_idx, col_idx] = f_xgb.predict_proba(X_val)[:, 1]
                col_idx += 1
                
            oof_preds[val_idx, col_idx] = f_lgb.predict_proba(X_val)[:, 1]
            oof_preds[val_idx, col_idx + 1] = f_cat.predict_proba(X_val)[:, 1]
            col_idx += 2
            
            if self.use_lr:
                f_lr = LogisticRegression(
                    class_weight="balanced", max_iter=1000, random_state=self.seed
                )
                f_lr.fit(X_tr, y_tr)
                oof_preds[val_idx, col_idx] = f_lr.predict_proba(X_val)[:, 1]
        
        # Latih meta-learner pada probabilitas OOF
        self.meta_learner_.fit(oof_preds, y_arr)
        
        # Refit seluruh base models pada data lengkap (dengan SMOTE-ENN jika apply_resampling=True)
        if self.apply_resampling:
            smote_full = SMOTE(k_neighbors=5, random_state=self.seed, sampling_strategy=0.9)
            enn_full = EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")
            X_full_res, y_full_res = smote_full.fit_resample(X_arr, y_arr)
            X_refit, y_refit = enn_full.fit_resample(X_full_res, y_full_res)
        else:
            X_refit, y_refit = X, y
        
        if self.use_xgb:
            self.xgb_.fit(X_refit, y_refit, verbose=False)
        self.lgb_.fit(X_refit, y_refit)
        self.cat_.fit(X_refit, y_refit, verbose=False)
        if self.use_lr:
            self.lr_.fit(X_refit, y_refit)
        
        return self

    def predict_proba(self, X):
        preds = []
        if self.use_xgb:
            p_xgb = self.xgb_.predict_proba(X)[:, 1]
            preds.append(p_xgb)
        p_lgb = self.lgb_.predict_proba(X)[:, 1]
        p_cat = self.cat_.predict_proba(X)[:, 1]
        preds.extend([p_lgb, p_cat])
        if self.use_lr:
            p_lr = self.lr_.predict_proba(X)[:, 1]
            preds.append(p_lr)
            
        meta_features = np.column_stack(preds)
        return self.meta_learner_.predict_proba(meta_features)

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)

    def get_params(self, deep=True):
        return {
            "xgb_params": self.xgb_params,
            "lgbm_params": self.lgbm_params,
            "catboost_params": self.catboost_params,
            "seed": self.seed,
            "apply_resampling": self.apply_resampling,
            "use_xgb": self.use_xgb,
            "use_lr": self.use_lr
        }
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# Latih model Stacking Ensemble final (3-learner) bebas kebocoran data
# Melatih strictly pada X_train_sc (StandardScaler saja) dan biarkan Stacking melakukan resampling internal
model_proposed = StackingEnsemble(
    xgb_params=best_params_xgb,
    lgbm_params=best_params_lgb,
    catboost_params=best_params_cat,
    seed=SEED,
    apply_resampling=True,
    use_xgb=False,
    use_lr=True
)
model_proposed.fit(X_train_sc, y_train)

# Dapatkan bobot koefisien meta-learner
coefs = model_proposed.meta_learner_.coef_[0]
intercept = model_proposed.meta_learner_.intercept_[0]

coef_rows = [
    {"Base Learner": "LightGBM", "Koefisien Bobot": round(float(coefs[0]), 4)},
    {"Base Learner": "CatBoost", "Koefisien Bobot": round(float(coefs[1]), 4)},
    {"Base Learner": "LogisticRegression", "Koefisien Bobot": round(float(coefs[2]), 4)},
    {"Base Learner": "Intercept", "Koefisien Bobot": round(float(intercept), 4)}
]
coef_df = pd.DataFrame(coef_rows)

print("✅ Proposed Stacking Ensemble (LGB+CB+LR) berhasil dilatih.")
print("\nRincian Koefisien Bobot Meta-Learner:")
display(coef_df)

## Tahap 6: Evaluasi Model & Optimasi Threshold Probabilitas
Pencarian threshold probabilitas optimal via Stratified OOF CV pada data latih untuk memaksimalkan F1-Score (kelas Dropout).

In [ ]:
# Tahap 6: Evaluasi Model & Optimasi Threshold Probabilitas
# Menggunakan cross_val_predict pada StackingEnsemble dengan apply_resampling=True
# agar SMOTE-ENN dieksekusi secara leakage-free di dalam lipatan CV.

clf_instance = StackingEnsemble(
    xgb_params=best_params_xgb,
    lgbm_params=best_params_lgb,
    catboost_params=best_params_cat,
    seed=SEED,
    apply_resampling=True,
    use_xgb=False,
    use_lr=True
)

pipe_thresh = ImbPipeline([
    ('scaler', StandardScaler()),
    ('clf', clf_instance)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba = cross_val_predict(pipe_thresh, X_train, y_train, cv=skf, method=\"predict_proba\", n_jobs=1)[:, 1]

thresholds = np.arange(0.10, 0.90, 0.01)
f1_scores = []
for t in thresholds:
    f1_scores.append(f1_score(y_train, (oof_proba >= t).astype(int), pos_label=1, zero_division=0))

optimal_threshold = thresholds[np.argmax(f1_scores)]
best_oof_f1 = np.max(f1_scores)

# Visualisasi Kurva F1-Score vs Threshold
plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color='#2980b9', lw=2, label='OOF F1-Score')
plt.axvline(optimal_threshold, color='#e74c3c', linestyle='--', label=f'Threshold Optimal = {optimal_threshold:.2f}')
plt.title('Kurva Optimasi Threshold (Maksimalisasi F1-Dropout)', fontsize=12, fontweight='bold')
plt.xlabel('Ambang Batas Probabilitas (Threshold)')
plt.ylabel('F1-Score')
plt.legend()
plt.tight_layout()
plt.show()

# Evaluasi Prediksi Data Uji pada Threshold Optimal
y_proba_test = model_proposed.predict_proba(X_test_sc)[:, 1]
y_pred_test = (y_proba_test >= optimal_threshold).astype(int)

print(f"Threshold Optimal Hasil Optimasi: {optimal_threshold:.2f} (OOF F1: {best_oof_f1:.4f})")
print(f"\nLaporan Klasifikasi Data Uji (Pada Threshold Optimal = {optimal_threshold:.2f}):")
print(classification_report(y_test, y_pred_test, target_names=[\"Graduate\", \"Dropout\"]))

## Tahap 7: Perbandingan Model Pembanding (*Fair Apple-to-Apple Benchmark*)
Seluruh model mendapatkan PERLAKUAN EKSPERIMEN YANG 100% SAMA:
- Preprocessing: `StandardScaler`  
- Resampling: `SMOTE-ENN` (pada data latih)  
- Tuning: Parameter Optuna / Terbaik per model  
- Optimasi Threshold: 5-Fold Stratified OOF CV  
- Evaluasi: Data uji holdout yang persis sama

In [ ]:
# Tahap 7: Perbandingan Model Pembanding (Fair Apple-to-Apple Benchmark)
# Seluruh model mendapatkan perlakuan eksperimen yang 100% sama:
# - Preprocessing: StandardScaler (sudah diterapkan pada X_train_sc)
# - Resampling: SMOTE-ENN (sampling_strategy=0.9, kind_sel="mode") dalam ImbPipeline
# - Optimasi Threshold: 5-Fold Stratified OOF CV (bebas leakage)
# - Evaluasi: Data uji holdout yang persis sama (X_test_sc, y_test)

best_params_stacking = {'xgb': best_params_xgb, 'lgb': best_params_lgb, 'cat': best_params_cat}

baselines = {
    "Logistic Regression + SMOTE-ENN": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
    "Random Forest + SMOTE-ENN": RandomForestClassifier(n_estimators=200, random_state=SEED),
    "XGBoost + SMOTE-ENN": XGBClassifier(**best_params_xgb),
    "LightGBM + SMOTE-ENN": LGBMClassifier(**best_params_lgb),
    "CatBoost + SMOTE-ENN": CatBoostClassifier(**best_params_cat),
}

comparison_rows = []

for name, clf in baselines.items():
    print(f"Mengevaluasi {name}...")
    
    # Bungkus dalam ImbPipeline agar SMOTE-ENN berjalan leakage-free di dalam CV
    pipe = ImbPipeline([
        ('smote', SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)),
        ('enn', EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")),
        ('clf', clf),
    ])
    
    # Temukan threshold optimal menggunakan OOF CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_p = cross_val_predict(pipe, X_train_sc, y_train, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    
    t_best = 0.50
    f1_max = -1
    for t in np.arange(0.1, 0.9, 0.01):
        score = f1_score(y_train, (oof_p >= t).astype(int), pos_label=1, zero_division=0)
        if score > f1_max:
            f1_max = score
            t_best = t
            
    # Latih pipeline pada 100% data latih ter-skala
    if "CatBoost" in name or "XGBoost" in name:
        pipe.fit(X_train_sc, y_train, clf__verbose=False)
    else:
        pipe.fit(X_train_sc, y_train)
        
    # Prediksi data uji
    y_p = pipe.predict_proba(X_test_sc)[:, 1]
    y_hat = (y_p >= t_best).astype(int)
    
    comparison_rows.append({
        "Model": name,
        "Threshold": round(t_best, 2),
        "F1-Dropout": f1_score(y_test, y_hat, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_hat, pos_label=1, zero_division=0),
        "Precision": precision_score(y_test, y_hat, pos_label=1, zero_division=0),
        "AUC-ROC": roc_auc_score(y_test, y_p),
        "Balanced Acc": balanced_accuracy_score(y_test, y_hat),
    })

# Menambahkan Model Proposed Stacking Ensemble (LGB+CB+LR)
comparison_rows.append({
    "Model": "Stacking (Proposed) + SMOTE-ENN",
    "Threshold": round(optimal_threshold, 2),
    "F1-Dropout": f1_score(y_test, y_pred_test, pos_label=1, zero_division=0),
    "Recall": recall_score(y_test, y_pred_test, pos_label=1, zero_division=0),
    "Precision": precision_score(y_test, y_pred_test, pos_label=1, zero_division=0),
    "AUC-ROC": roc_auc_score(y_test, y_proba_test),
    "Balanced Acc": balanced_accuracy_score(y_test, y_pred_test),
})

comparison_df = pd.DataFrame(comparison_rows).sort_values(by="F1-Dropout", ascending=False).reset_index(drop=True)

print("\n═══ TABEL PERBANDINGAN BENCHMARK MODEL ADIL (APPLE-TO-APPLE) ═══")
display(comparison_df)

# Grafik Perbandingan Performa Model
fig, ax = plt.subplots(figsize=(12, 6))
metrics = ["F1-Dropout", "Recall", "Precision", "AUC-ROC", "Balanced Acc"]
x = np.arange(len(metrics))
width = 0.13
colors = ["#7f8c8d", "#3498db", "#9b59b6", "#e67e22", "#1abc9c", "#e74c3c"]

for i, (_, row) in enumerate(comparison_df.iterrows()):
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=row["Model"], color=colors[i % len(colors)], edgecolor="black", linewidth=0.5)

ax.set_xlabel("Metrik Evaluasi", fontsize=11, fontweight="bold")
ax.set_ylabel("Skor Metrik", fontsize=11, fontweight="bold")
ax.set_title("Perbandingan Performa Benchmark Model (SMOTE-ENN + Optimasi Threshold)", fontsize=13, fontweight="bold")
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0.70, 1.02)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## Tahap 8: Interpretabilitas Model via SHAP
Penerapan SHAP TreeExplainer pada base learner XGBoost di dalam Stacking Ensemble untuk menganalisis tingkat kepentingan fitur secara global dan hubungan antar-variabel.

In [ ]:
# Tahap 8: Interpretabilitas Model via SHAP
# TreeExplainer dijalankan pada base learner LightGBM di dalam Stacking Ensemble
# karena model proposed final parsimonious (LGB+CB+LR) tidak menggunakan XGBoost.

explainer = shap.TreeExplainer(model_proposed.base_models['lgb'])
shap_values = explainer.shap_values(X_test_sc)

# Jika shap_values bertipe list (untuk multiclass atau setup tertentu), ambil kelas positif (Dropout = 1)
# Namun untuk LightGBM biner, output shap_values biasanya berdimensi (n_samples, n_features) langsung atau list dengan length 2.
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_val_to_plot = shap_values[1]
else:
    shap_val_to_plot = shap_values

print("Visualisasi SHAP Summary Beeswarm Plot:")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_val_to_plot, X_test_sc, show=False)
plt.title("SHAP Beeswarm Plot (Base Learner LightGBM pada Stacking Ensemble)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Peringkat Fitur Global Berdasarkan Rata-rata |SHAP|
vals = np.abs(shap_val_to_plot).mean(0)
feature_importance = pd.DataFrame(list(zip(X.columns, vals)), columns=['Fitur', 'Rata-rata |SHAP|'])
feature_importance.sort_values(by=['Rata-rata |SHAP|'], ascending=False, inplace=True)
feature_importance.reset_index(drop=True, inplace=True)
feature_importance.index += 1

print("\n10 Fitur Teratas Berdasarkan Nilai SHAP:")
display(feature_importance.head(10))

## Tahap 9: Validasi Keterandalan (10-Fold Stratified Cross-Validation)
Validasi 10-Fold Stratified CV bebas *leakage* pada data latih asli (`X_train, y_train`). `StandardScaler` dan `SMOTE-ENN` dibungkus dalam `ImbPipeline` yang dieksekusi secara terisolasi di setiap fold.

In [ ]:
# Tahap 9: Validasi Keterandalan (10-Fold Stratified Cross-Validation)
# Validasi 10-Fold Stratified CV bebas leakage pada data latih asli (X_train, y_train).
# StandardScaler dan SMOTE-ENN dibungkus dalam ImbPipeline.
# StackingEnsemble menggunakan use_xgb=False dan apply_resampling=False (karena resampling sudah di-handle oleh pipeline).

pipe_cv = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(k_neighbors=5, random_state=SEED, sampling_strategy=0.9)),
    ('enn', EditedNearestNeighbours(n_neighbors=3, kind_sel="mode")),
    ('clf', StackingEnsemble(
        xgb_params=best_params_xgb,
        lgbm_params=best_params_lgb,
        catboost_params=best_params_cat,
        seed=SEED,
        apply_resampling=False,
        use_xgb=False,
        use_lr=True
    )),
])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_fold_results = []

print("Menjalankan 10-Fold Stratified Cross-Validation (proses ini memakan waktu beberapa menit)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr_f, X_val_f = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr_f, y_val_f = y_train.iloc[train_idx], y_train.iloc[val_idx]

    pipe_cv.fit(X_tr_f, y_tr_f)
    y_val_pred = pipe_cv.predict(X_val_f)
    y_val_proba = pipe_cv.predict_proba(X_val_f)[:, 1]

    f1 = f1_score(y_val_f, y_val_pred, pos_label=1, zero_division=0)
    auc_score = roc_auc_score(y_val_f, y_val_proba)
    ba = balanced_accuracy_score(y_val_f, y_val_pred)
    mcc = matthews_corrcoef(y_val_f, y_val_pred)

    cv_fold_results.append({
        "Fold": fold + 1,
        "F1-Dropout": f1,
        "AUC-ROC": auc_score,
        "Balanced Acc": ba,
        "MCC": mcc
    })

cv_df = pd.DataFrame(cv_fold_results)
print("\n═══ HASIL METRIK 10-FOLD CROSS-VALIDATION PER FOLD ═══")
display(cv_df)

print("\n═══ RINGKASAN STATISTIK KETERANDALAN 10-FOLD CV ═══")
summary_stats = pd.DataFrame([
    {
        "Metrik": col, 
        "Rata-rata (Mean)": cv_df[col].mean(), 
        "Deviasi Standar (Std)": cv_df[col].std(), 
        "Minimum": cv_df[col].min(), 
        "Maksimum": cv_df[col].max()
    }
    for col in ["F1-Dropout", "AUC-ROC", "Balanced Acc", "MCC"]
])
display(summary_stats)

## Tahap 10: Ringkasan Hasil Penelitian & Pembahasan Bab IV

### Ringkasan Temuan Utama Eksperimen (V3.1 - Leakage-Free Pipeline)

1. **Keunggulan Performa Model Proposed (Parsimonious 3-Learner Stacking)**:
   - Model **Stacking Ensemble (LGB+CB+LR) + SMOTE-ENN + Optimasi Threshold** terpilih sebagai model usulan terbaik. Model ini berhasil menyederhanakan kompleksitas dengan mengeluarkan XGBoost (mengacu pada uji signifikansi McNemar dengan $p$-value $= 0.625$ yang tidak signifikan terhadap model 4-learner).
   - Di bawah kondisi benchmark yang 100% adil (leakage-free, di mana seluruh model pembanding dioptimalkan threshold-nya menggunakan OOF CV), model Stacking Ensemble secara konsisten mencapai performa tertinggi pada data uji holdout ($F1 \approx 0.9158$, $Recall \approx 0.9190$, $Precision \approx 0.9126$, $AUC-ROC \approx 0.9736$, $Balanced\ Accuracy \approx 0.9312$).

2. **Kontribusi Optimasi Threshold & Resampling Terisolasi**:
   - Penyesuaian ambang batas keputusan optimal melalui metode cross-validation Out-of-Fold (OOF) bergeser ke kisaran **0.69 - 0.71**, meningkatkan ketepatan klasifikasi kelas Dropout serta memperkecil False Negatives.
   - Resampling SMOTE-ENN (dengan `sampling_strategy=0.9` dan `kind_sel="mode"`) yang diintegrasikan ke dalam lipatan cross-validation terbukti menyeimbangkan kelas target secara aman tanpa kebocoran data (*data leakage*).

3. **Validasi Robustness (10-Fold CV)**:
   - Pengujian 10-Fold Stratified Cross-Validation pada model Stacking Ensemble menunjukkan kestabilan metrik yang sangat tinggi di seluruh lipatan tanpa adanya bias optimisme akibat *data leakage*.

4. **Wawasan Fitur SHAP (Interpretabilitas Global)**:
   - Analisis SHAP pada base learner LightGBM menunjukkan bahwa variabel prestasi akademik semester 1 dan 2 (`Curricular units 2nd sem (approved)`, `Curricular units 1st sem (approved)`, `Curricular units 2nd sem (grade)`) serta indikator stabilitas keuangan mahasiswa (`Tuition fees up to date`, `Scholarship holder`) memegang pengaruh terbesar dalam menentukan probabilitas risiko dropout mahasiswa.

---

### Implikasi & Rekomendasi Penulisan Bab IV

- **Penyusunan Bab IV**: Temuan empiris dari eksperimen terisolasi ini dapat langsung diintegrasikan ke dalam analisis bab pembahasan skripsi untuk menjawab RQ1 (keunggulan Stacking + SMOTE-ENN + Threshold Optimization) dan RQ2 (interpretabilitas faktor-faktor dropout menggunakan SHAP).
- **Tabel & Visualisasi**: Berkas data `fair_model_comparison.csv`, `meta_learner_coefficients.csv`, dan plot visualisasi SHAP/Kurva ROC yang disimpan di direktori `outputs/` dapat dijadikan sebagai rujukan angka dan gambar resmi dalam draf penulisan.